<a href="https://colab.research.google.com/github/inoue0426/llm-tuning-playground/blob/main/notebooks/04_med_refl_dpo_before_after.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/inoue0426/llm-tuning-playground/blob/main/notebooks/04_med_refl_dpo_before_after.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 - Medical DPO with Before/After evaluation

This notebook uses the public **HANI-LAB/Med-REFL-DPO** dataset. It is an English medical DPO dataset released under Apache-2.0. The dataset contains preference pairs designed to improve medical reasoning and reflection.

The main goal here is different from Notebook 03: measure the same held-out preference set **before and after DPO**.

Pipeline: **Base -> evaluate -> LoRA + DPO -> evaluate -> compare**.

## Dataset choice

`HANI-LAB/Med-REFL-DPO` is an English medical preference dataset with Apache-2.0 licensing. Its `reasoning_enhancement` subset contains medical questions paired with a preferred reasoning trajectory and a less-preferred trajectory. The full dataset is much larger than what we need for a first Colab experiment, so we use a reproducible subset.

In [1]:
!pip -q install -U \
    "transformers>=4.55,<5" \
    "datasets>=3.6,<5" \
    "peft>=0.17,<1" \
    "trl>=0.21,<1" \
    "accelerate>=1.10,<2" \
    "bitsandbytes>=0.46,<1" \
    "torchao>=0.16,<1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 139.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 50.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import torch
import transformers, datasets, peft, trl

print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('Datasets:', datasets.__version__)
print('PEFT:', peft.__version__)
print('TRL:', trl.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab before running this notebook.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

PyTorch: 2.11.0+cu128
Transformers: 4.57.6
Datasets: 4.8.5
PEFT: 0.20.0
TRL: 0.29.1
CUDA available: True
GPU: NVIDIA L4
VRAM (GB): 22.0


## 1. Load and sample the medical preference data

We use `reasoning_enhancement`. For a first L4 experiment, 800 training pairs and 100 held-out pairs are enough to make the before/after comparison inexpensive. The split is created before selecting the subset, so evaluation rows are never used for training.

In [3]:
from datasets import load_dataset

DATASET_NAME = 'HANI-LAB/Med-REFL-DPO'
CONFIG_NAME = 'reasoning_enhancement'
SEED = 42
MAX_TRAIN = 800
MAX_EVAL = 100

raw = load_dataset(DATASET_NAME, CONFIG_NAME, split='train')
raw = raw.shuffle(seed=SEED)

eval_dataset_raw = raw.select(range(MAX_EVAL))
train_dataset_raw = raw.select(range(MAX_EVAL, MAX_EVAL + MAX_TRAIN))

print('Source rows:', len(raw))
print('Train rows:', len(train_dataset_raw))
print('Eval rows:', len(eval_dataset_raw))
print('Columns:', train_dataset_raw.column_names)
print('Example instruction:', train_dataset_raw[0]['instruction'][:300])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/Med-REFL-Data/Reasoning Enhancement(…):   0%|          | 0.00/158M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21326 [00:00<?, ? examples/s]

Source rows: 21326
Train rows: 800
Eval rows: 100
Columns: ['instruction', 'input', 'chosen', 'rejected']
Example instruction: A 26-year-old primigravid woman at 39 weeks' gestation is admitted to the hospital in active labor. Pregnancy was complicated by mild oligohydramnios detected a week ago, which was managed with hydration. Her pulse is 92/min, respirations are 18/min, and blood pressure is 134/76 mm Hg. Pelvic examin


## 2. Convert to TRL conversational preference format

Qwen's chat template is used consistently for both the prompt and the completions.

In [4]:
from datasets import Dataset

def to_conversational(row):
    prompt = str(row['instruction']).strip()
    if row.get('input', ''):
        extra = str(row['input']).strip()
        if extra:
            prompt = prompt + '\n\n' + extra
    return {
        'prompt': [{'role': 'user', 'content': prompt}],
        'chosen': [{'role': 'assistant', 'content': str(row['chosen']).strip()}],
        'rejected': [{'role': 'assistant', 'content': str(row['rejected']).strip()}],
    }

train_dataset = Dataset.from_list([to_conversational(x) for x in train_dataset_raw])
eval_dataset = Dataset.from_list([to_conversational(x) for x in eval_dataset_raw])

print(train_dataset[0])

{'prompt': [{'role': 'user', 'content': "A 26-year-old primigravid woman at 39 weeks' gestation is admitted to the hospital in active labor. Pregnancy was complicated by mild oligohydramnios detected a week ago, which was managed with hydration. Her pulse is 92/min, respirations are 18/min, and blood pressure is 134/76 mm Hg. Pelvic examination shows 100% cervical effacement and 10 cm cervical dilation; the vertex is at 0 station. Cardiotocography is shown. Which of the following is the most appropriate next step in management?\nthe options are:{'A': 'Emergent cesarean section', 'B': 'Reassurance', 'C': 'Maternal repositioning and oxygen administration', 'D': 'Elevation of the fetal head', 'E': 'Rapid amnioinfusion'}"}], 'chosen': [{'role': 'assistant', 'content': "## Thinking\n\nHere's an interesting medical case about a 26-year-old primigravid woman at 39 weeks' gestation who's admitted in active labor. Let me think... She had mild oligohydramnios a week ago, which they managed with 

## 3. Load Qwen2.5-0.5B-Instruct

We keep the same base model as Notebooks 01-03 so the effect of the dataset and DPO procedure is easier to interpret.

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype)
model = model.cuda()

print('Has chat template:', tokenizer.chat_template is not None)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Has chat template: True


## 4. Preference evaluation function

We score `chosen` and `rejected` directly under the model and calculate:

`margin = mean_logP(chosen | prompt) - mean_logP(rejected | prompt)`

Preference accuracy is the fraction of held-out pairs where the margin is positive. The exact same evaluation set is used before and after DPO.

In [6]:
import torch.nn.functional as F

def completion_logprob(model, tokenizer, prompt_messages, completion_messages):
    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    )[0]

    full_ids = tokenizer.apply_chat_template(
        prompt_messages + completion_messages,
        tokenize=True,
        add_generation_prompt=False,
        return_tensors='pt',
    )[0]

    prefix_len = len(prompt_ids)
    if not torch.equal(full_ids[:prefix_len], prompt_ids):
        raise ValueError('Prompt is not an exact token prefix. Check the chat template.')

    input_ids = full_ids.unsqueeze(0).to(model.device)
    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
    labels = input_ids[:, 1:]
    token_log_probs = log_probs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    completion_log_probs = token_log_probs[:, prefix_len - 1:]
    return completion_log_probs.mean().item()

def evaluate_preferences(model, tokenizer, dataset):
    model.eval()
    rows = []
    for example in dataset:
        chosen_lp = completion_logprob(model, tokenizer, example['prompt'], example['chosen'])
        rejected_lp = completion_logprob(model, tokenizer, example['prompt'], example['rejected'])
        margin = chosen_lp - rejected_lp
        rows.append({
            'chosen_logprob': chosen_lp,
            'rejected_logprob': rejected_lp,
            'margin': margin,
            'correct': margin > 0,
        })

    accuracy = sum(r['correct'] for r in rows) / len(rows)
    mean_chosen = sum(r['chosen_logprob'] for r in rows) / len(rows)
    mean_rejected = sum(r['rejected_logprob'] for r in rows) / len(rows)
    mean_margin = sum(r['margin'] for r in rows) / len(rows)

    return {
        'accuracy': accuracy,
        'mean_chosen_logprob': mean_chosen,
        'mean_rejected_logprob': mean_rejected,
        'mean_margin': mean_margin,
        'rows': rows,
    }

## 5. BEFORE DPO

This is the baseline. Run it before attaching or training any LoRA adapter.

In [7]:
before = evaluate_preferences(model, tokenizer, eval_dataset)
print(f"Preference accuracy : {before['accuracy']:.3f}")
print(f"Mean chosen log-prob: {before['mean_chosen_logprob']:.3f}")
print(f"Mean rejected log-prob: {before['mean_rejected_logprob']:.3f}")
print(f"Mean margin         : {before['mean_margin']:.3f}")

Preference accuracy : 0.480
Mean chosen log-prob: -1.626
Mean rejected log-prob: -1.627
Mean margin         : 0.002


## 6. BEFORE DPO qualitative generation

We keep one held-out medical question fixed so the qualitative comparison is also before/after.

In [8]:
def generate(model, tokenizer, question, max_new_tokens=160):
    messages = [{'role': 'user', 'content': question}]
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True,
    )
    encoded = {k: v.to(model.device) for k, v in encoded.items()}
    with torch.no_grad():
        output = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

TEST_INDEX = 0
test_question = eval_dataset[TEST_INDEX]['prompt'][0]['content']
print('QUESTION:', test_question)
print()
print('BEFORE DPO:')
print(generate(model, tokenizer, test_question))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUESTION: A 12-year-old boy and his mother are brought to the emergency department after a motor vehicle accident. The boy was an unrestrained passenger in a head-on collision and was ejected from the front seat. The patient's mother was the driver and she is currently being resuscitated. Neither the child nor the mother are conscious; however, it is documented that the family are all Jehovah's witnesses and would not want a transfusion in an acute situation. The husband/father arrives to the trauma bay and confirms this wish that everyone in the family would not want a transfusion in accordance with their beliefs. The father is confirmed as the official healthcare proxy. Which of the following is the best next step in management?
the options are:{'A': 'Consult the hospital ethics committee', 'B': 'Do not transfuse the boy and transfuse the mother', 'C': 'Do not transfuse the boy or the mother', 'D': 'Do not transfuse the mother and transfuse the boy', 'E': 'Transfuse the boy and mothe

## 7. LoRA + DPO

We train only a small LoRA adapter. `max_prompt_length` is intentionally omitted for compatibility with the TRL 0.29.x environment used in the earlier notebooks.

In [9]:
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)

dpo_args = DPOConfig(
    output_dir='./outputs/med-refl-dpo',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=1e-5,
    beta=0.1,
    max_length=1024,
    logging_steps=10,
    save_strategy='no',
    report_to='none',
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
)

trainer = DPOTrainer(
    model=model,
    args=dpo_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.train()

Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.709700
20,0.681100
30,0.690600
40,0.658800
50,0.673100
60,0.668400
70,0.634600
80,0.637800
90,0.636500
100,0.645500


TrainOutput(global_step=100, training_loss=0.6636017417907715, metrics={'train_runtime': 428.9787, 'train_samples_per_second': 1.865, 'train_steps_per_second': 0.233, 'total_flos': 3374052214697472.0, 'train_loss': 0.6636017417907715})

## 8. AFTER DPO

Use the exact same evaluation set and metric as in Section 5. This is the key experiment.

In [10]:
after = evaluate_preferences(model, tokenizer, eval_dataset)
print(f"Preference accuracy : {after['accuracy']:.3f}")
print(f"Mean chosen log-prob: {after['mean_chosen_logprob']:.3f}")
print(f"Mean rejected log-prob: {after['mean_rejected_logprob']:.3f}")
print(f"Mean margin         : {after['mean_margin']:.3f}")

Preference accuracy : 0.500
Mean chosen log-prob: -1.623
Mean rejected log-prob: -1.627
Mean margin         : 0.004


## 9. BEFORE vs AFTER

The most useful quantities are the preference accuracy and mean margin. An increase means the DPO model separates the preferred response from the rejected response more strongly on the held-out pairs.

In [11]:
print('Metric comparison')
print('-' * 70)
print(f"{'Metric':30s} {'Before':>12s} {'After':>12s} {'Delta':>12s}")
print('-' * 70)
for label, key in [
    ('Preference accuracy', 'accuracy'),
    ('Mean chosen log-prob', 'mean_chosen_logprob'),
    ('Mean rejected log-prob', 'mean_rejected_logprob'),
    ('Mean margin', 'mean_margin'),
]:
    b = before[key]
    a = after[key]
    print(f"{label:30s} {b:12.3f} {a:12.3f} {a-b:12.3f}")

Metric comparison
----------------------------------------------------------------------
Metric                               Before        After        Delta
----------------------------------------------------------------------
Preference accuracy                   0.480        0.500        0.020
Mean chosen log-prob                 -1.626       -1.623        0.002
Mean rejected log-prob               -1.627       -1.627       -0.000
Mean margin                           0.002        0.004        0.003


## 10. AFTER DPO qualitative generation

The prompt is identical to the BEFORE DPO generation above. Do not treat one generation as a factual benchmark; use it only as a qualitative sanity check.

In [12]:
model.eval()
print('QUESTION:', test_question)
print()
print('AFTER DPO:')
print(generate(model, tokenizer, test_question))

QUESTION: A 12-year-old boy and his mother are brought to the emergency department after a motor vehicle accident. The boy was an unrestrained passenger in a head-on collision and was ejected from the front seat. The patient's mother was the driver and she is currently being resuscitated. Neither the child nor the mother are conscious; however, it is documented that the family are all Jehovah's witnesses and would not want a transfusion in an acute situation. The husband/father arrives to the trauma bay and confirms this wish that everyone in the family would not want a transfusion in accordance with their beliefs. The father is confirmed as the official healthcare proxy. Which of the following is the best next step in management?
the options are:{'A': 'Consult the hospital ethics committee', 'B': 'Do not transfuse the boy and transfuse the mother', 'C': 'Do not transfuse the boy or the mother', 'D': 'Do not transfuse the mother and transfuse the boy', 'E': 'Transfuse the boy and mothe

## 11. Save the LoRA adapter

Only the adapter is saved.

In [13]:
ADAPTER_DIR = './outputs/med-refl-dpo-adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved:', ADAPTER_DIR)

Saved: ./outputs/med-refl-dpo-adapter


## Interpretation

- If accuracy and margin increase: DPO learned the held-out preference signal more strongly.
- If accuracy was already near 1.0 before DPO: the base model already separated the pairs, so a large gain is unlikely.
- This experiment measures preference alignment, not comprehensive medical factuality.
- The next stronger experiment is to evaluate factual correctness on a separate medical QA benchmark that was not used to construct the preference data.